In [ ]:
import os
from paths import CLINVAR_CSV, CONFIG, DB_DIR, OUTPUT_DIR, SCORES_DIR

import pandas as pd
from aigct.container import VEBenchmarkContainer
container = VEBenchmarkContainer(CONFIG)

query_mgr = container.query_mgr


In [ ]:
tasks_df = query_mgr.get_tasks()

In [ ]:
import pandas as pd
genes = pd.read_csv(os.path.join(DB_DIR, "DDD", "variant_filter_gene.csv"))
genes = genes[genes['FILTER_CODE'] == 'DDD_RELATED_GENES_PRIMATEAI']
genes = genes['GENE_SYMBOL'].to_list()

In [ ]:
from aigct.model import VEQueryCriteria
clinvar = pd.read_csv(CLINVAR_CSV)
qry1 = VEQueryCriteria(variant_ids = clinvar , include_variant_ids= False)

In [ ]:
from aigct.model import VEQueryCriteria
qry1 = VEQueryCriteria(filter_names=['DDD_case'], gene_symbols= genes)

In [ ]:
CLINV = query_mgr.get_variants_by_task('DDD', qry1)
print(len(CLINV[CLINV['BINARY_LABEL'] == 0]), len(CLINV[CLINV['BINARY_LABEL'] == 1]))

In [ ]:
qry2 = VEQueryCriteria(filter_names=['DDD_control1', 'DDD_control2', 'DDD_control3', 'DDD_control4'])

In [ ]:
from aigct.container import VEBenchmarkContainer

container = VEBenchmarkContainer(CONFIG)

query_mgr = container.query_mgr
CHD = query_mgr.get_variants_by_task('DDD', qry1)
CHD.to_csv(os.path.join(OUTPUT_DIR, "DDD_pri_EVE.csv"))

In [ ]:
DDD_case = query_mgr.get_variants_by_task('DDD', qry1)
DDD_control = query_mgr.get_variants_by_task('DDD', qry2)

In [ ]:
DDD = pd.concat([DDD_case, DDD_control], ignore_index= True)

In [ ]:
DDD_MAVEN_EVE_ = pd.merge(DDD_MAVEN_EVE, DDD, right_on= ['GENOME_ASSEMBLY', 'CHROMOSOME', 'POSITION', 'REFERENCE_NUCLEOTIDE','ALTERNATE_NUCLEOTIDE'], left_on= ['GENOME_ASSEMBLY', 'CHROMOSOME', 'POSITION', 'REFERENCE_NUCLEOTIDE','ALTERNATE_NUCLEOTIDE'], how = 'inner', indicator= True)

In [ ]:
DDD_MAVEN_EVE_

In [ ]:
DDD_MAVEN_EVE

In [ ]:
DDD_MAVEN_EVE = pd.read_csv(os.path.join(SCORES_DIR, "DDD_MAVEN_EVE.csv"))
DDD_pri_EVE = pd.read_csv(os.path.join(SCORES_DIR, "DDD_pri_EVE.csv"))
DDD_EVE = pd.concat([DDD_MAVEN_EVE, DDD_pri_EVE])

In [ ]:
metrics = container.analyzer.compute_metrics(
    "DDD", DDD_MAVEN_EVE_,'EVE', variant_effect_sources= ['MAVEN', 'MAVENAVG', 'ALPHAM', 'REVEL', 'ESM1B'], vep_min_overlap_percent=1, variant_vep_retention_percent=100)

In [ ]:
metrics = container.analyzer.compute_metrics(
    "DDD", 
    variant_query_criteria = qry1, 
    vep_min_overlap_percent=90, 
    variant_vep_retention_percent=100)

In [ ]:
container.reporter.write_summary(metrics)

In [ ]:
container.plotter.plot_results(metrics)


In [ ]:
vep_stats_df = query_mgr.get_variant_effect_source_stats("CANCER",
    ["ALPHAM", "REVEL", "EVE"])

In [ ]:
vep_stats_df